In [1]:
import cv2
import numpy as np
import pickle
from ultralytics import YOLO
from collections import defaultdict
import yaml

# ── 모델 로드 ──────────────────────────────────────────
yolo = YOLO(r'C:\Users\neo62\sperm-ai\models\yolo11_sperm_v2\weights\best.pt')

with open(r'C:\Users\neo62\sperm-ai\models\motility_regressor.pkl', 'rb') as f:
    reg_data = pickle.load(f)

reg_model  = reg_data['model']
scaler     = reg_data['scaler']
feat_cols  = reg_data['feature_cols']
config_path = r'C:\Users\neo62\sperm-ai\bytetrack_custom.yaml'

print("✅ 모델 로드 완료")

# ── 핵심 함수: 영상 하나 → 운동성 결과 ───────────────────
def analyze_video(video_path):
    """영상 입력 → 운동성 % 출력"""

    # Step 1: 전체 정자 수 N (첫 10프레임 중앙값)
    cap = cv2.VideoCapture(video_path)
    counts = []
    for _ in range(10):
        ret, frame = cap.read()
        if not ret: break
        res = yolo(frame, verbose=False, conf=0.3)
        counts.append(int((res[0].boxes.cls == 0).sum()))
    cap.release()
    N = int(np.median(counts)) if counts else 0

    # Step 2: ByteTrack 추적
    cap = cv2.VideoCapture(video_path)
    track_history = defaultdict(list)
    fps = cap.get(cv2.CAP_PROP_FPS)
    max_frames = min(int(fps * 5), 250)

    for fidx in range(max_frames):
        ret, frame = cap.read()
        if not ret: break
        res = yolo.track(frame, persist=True,
                         tracker=config_path,
                         verbose=False, conf=0.3)
        if res[0].boxes.id is not None:
            for box, tid, cls in zip(
                res[0].boxes.xywh.cpu().numpy(),
                res[0].boxes.id.cpu().numpy().astype(int),
                res[0].boxes.cls.cpu().numpy().astype(int)
            ):
                if cls == 0:
                    track_history[tid].append(
                        (fidx, float(box[0]), float(box[1])))
    cap.release()

    # Step 3: 특징 계산
    speeds, lins, straight_dists = [], [], []
    for tid, pts in track_history.items():
        if len(pts) < 5: continue
        coords = np.array([(cx, cy) for _, cx, cy in pts])
        dists = np.sqrt(np.sum(np.diff(coords, axis=0)**2, axis=1))
        total_dist = float(np.sum(dists))
        straight_dist = float(np.sqrt(
            (coords[-1][0]-coords[0][0])**2 +
            (coords[-1][1]-coords[0][1])**2))
        avg_speed = total_dist / len(pts)
        linearity = straight_dist / (total_dist + 1e-6)
        speeds.append(avg_speed)
        lins.append(linearity)
        straight_dists.append(straight_dist)

    if not speeds:
        return None

    speeds = np.array(speeds)
    feat = {
        'speed_mean':    float(np.mean(speeds)),
        'speed_median':  float(np.median(speeds)),
        'speed_75':      float(np.percentile(speeds, 75)),
        'speed_90':      float(np.percentile(speeds, 90)),
        'lin_mean':      float(np.mean(lins)),
        'lin_75':        float(np.percentile(lins, 75)),
        'straight_mean': float(np.mean(straight_dists)),
        'ratio_fast':    float(np.mean(speeds > 1.5)),
        'ratio_medium':  float(np.mean((speeds > 0.5) & (speeds <= 1.5))),
        'ratio_slow':    float(np.mean(speeds <= 0.5)),
        'n_tracks':      len(speeds),
    }

    # Step 4: 회귀 모델로 운동성 % 예측
    X = np.array([[feat[c] for c in feat_cols]])
    X_scaled = scaler.transform(X)
    pred = reg_model.predict(X_scaled)[0]

    # 합이 100%가 되도록 정규화
    pred = np.clip(pred, 0, 100)
    total = pred.sum()
    if total > 0:
        pred = pred / total * 100

    return {
        'N': N,
        'progressive':     round(pred[0], 1),
        'non_progressive': round(pred[1], 1),
        'immotile':        round(pred[2], 1),
    }

print("✅ 파이프라인 함수 준비 완료")

# ── VISEM-Tracking Val 4명으로 최종 검증 ──────────────
import pandas as pd

base = r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\VISEM_Tracking_Train_v4\Train'
csv  = r'C:\Users\neo62\sperm-ai\data\raw\VISEM-Tracking\semen_analysis_data_Train.csv'
df   = pd.read_csv(csv)

val_ids = ['11', '14', '22', '23']

print("\n=== 최종 파이프라인 검증 (Val 4명) ===")
print(f"{'참가자':<6} {'AI전진':>6} {'실제전진':>8} {'AI비전진':>8} "
      f"{'실제비전진':>10} {'AI비운동':>8} {'실제비운동':>10}")
print("-" * 62)

maes = []
for pid in val_ids:
    video_path = f'{base}\\{pid}\\{pid}.mp4'
    result = analyze_video(video_path)
    if result is None:
        continue

    row = df[df['ID'] == int(pid)]
    real_prog    = float(row['Progressive motility (%)'].values[0])
    real_non     = float(row['Non progressive sperm motility (%)'].values[0])
    real_imm     = float(row['Immotile sperm (%)'].values[0])

    mae = (abs(result['progressive'] - real_prog) +
           abs(result['non_progressive'] - real_non) +
           abs(result['immotile'] - real_imm)) / 3
    maes.append(mae)

    print(f"{pid:<6} {result['progressive']:>6.1f}% {real_prog:>7.1f}% "
          f"{result['non_progressive']:>7.1f}% {real_non:>9.1f}% "
          f"{result['immotile']:>7.1f}% {real_imm:>9.1f}%")

print("-" * 62)
print(f"\n전체 평균 MAE: {np.mean(maes):.1f}%p")
print(f"(이전: 15.1%p → 목표: ~7.3%p)")

✅ 모델 로드 완료
✅ 파이프라인 함수 준비 완료

=== 최종 파이프라인 검증 (Val 4명) ===
참가자      AI전진     실제전진    AI비전진      실제비전진    AI비운동      실제비운동
--------------------------------------------------------------
11       21.4%    11.0%    30.2%      17.0%    48.4%      72.0%
14       42.0%    41.0%    33.5%      43.0%    24.5%      16.0%
22       35.2%    56.0%    30.5%      25.0%    34.3%      19.0%
23       19.1%    18.0%    28.5%      34.0%    52.4%      48.0%
--------------------------------------------------------------

전체 평균 MAE: 9.9%p
(이전: 15.1%p → 목표: ~7.3%p)
